# Flow matching on synthetic Gaussian point clouds
Create independent training, validation and test splits, train a small generator, generate held-out
clouds using the package evaluation command, and compare every feature with the reference distribution.
The four target features have shifted means and unequal widths, so reproducing standard-normal input noise is insufficient.
The dataset name `gaussian_example` selects standard Gaussian base noise. All points are real (fixed multiplicity).
Install with `pip install -e ".[examples]"`. Set `OMNILEARNED_EXAMPLE_SMOKE=1` for a short execution check;
that mode is not a convergence test.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from omnilearned.metrics import binary_metrics

# Run from the checkout root or examples/. Install: pip install -e ".[examples]"
ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
DATA = Path(os.environ.get("OMNILEARNED_EXAMPLE_DATA", ROOT / "datasets")).resolve()
OUTPUT = Path(os.environ.get("OMNILEARNED_EXAMPLE_OUTPUT", ROOT / "example_outputs")).resolve()
CHECKPOINTS = OUTPUT / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get("OMNILEARNED_EXAMPLE_SMOKE", "0") == "1"

def run(*args):
    subprocess.run([sys.executable, "-m", "omnilearned.cli", *map(str, args)], check=True, cwd=ROOT)

In [ ]:
import h5py
import tempfile
# A fresh data directory also ensures existing immutable index snapshots are not reused for new data.
DATA = Path(tempfile.mkdtemp(prefix="gaussian_data_", dir=OUTPUT))
RESULTS = Path(tempfile.mkdtemp(prefix="gaussian_results_", dir=OUTPUT))
DATASET = "gaussian_example"
TAG = "gaussian_small"
MEAN = np.array([0.5, -0.7, 2.5, 3.0], dtype=np.float32)
STD = np.array([0.6, 0.8, 0.4, 0.5], dtype=np.float32)
PARTICLES = 8 if SMOKE else 16
rng = np.random.default_rng(42)
for split, n in [("train", 32 if SMOKE else 4096),
                 ("val", 8 if SMOKE else 512),
                 ("test", 16 if SMOKE else 512)]:
    folder = DATA / DATASET / split
    folder.mkdir(parents=True)
    x = (rng.normal(size=(n, PARTICLES, 4)) * STD + MEAN).astype(np.float32)
    # The loader interprets feature 2 == 0 as padding.
    x[..., 2][x[..., 2] == 0] = np.finfo(np.float32).eps
    with h5py.File(folder / "sample.h5", "w") as f:
        f["data"] = x
        f["pid"] = np.zeros(n, dtype=np.int64)
        # Last global feature is multiplicity / 100, even for an unconditional generator.
        f["global"] = np.full((n, 1), PARTICLES / 100, dtype=np.float32)

In [ ]:
args = ["train", "--dataset", DATASET, "--path", DATA,
        "--output_dir", CHECKPOINTS, "--save-tag", TAG,
        "--mode", "generator", "--size", "small", "--num-feat", 4,
        "--num-classes", 1, "--no-conditional", "--no-interaction",
        "--epoch", 1 if SMOKE else 30, "--batch", 4 if SMOKE else 64,
        "--lr", 1e-4, "--optim", "adam", "--ema", "--ema-decay", 0.99,
        "--num-workers", 0, "--no-check-remote", "--seed", 42]
if SMOKE:
    args += ["--iterations", 2, "--val-iterations", 1]
run(*args)

## Generate samples
Evaluation uses the held-out test multiplicities and class labels (all class 0), draws independent base noise,
and integrates from t=1 to t=0. Output HDF5 files retain the input schema and multiplicity.

In [ ]:
run("evaluate", "--dataset", DATASET, "--path", DATA,
    "--input_dir", CHECKPOINTS, "--output_dir", RESULTS, "--save-tag", TAG,
    "--mode", "generator", "--num-classes", 1, "--batch", 4 if SMOKE else 64,
    "--nsteps", 3 if SMOKE else 64, "--use-ema", "--num-workers", 0,
    "--no-check-remote", "--seed", 123)

In [ ]:
with h5py.File(DATA / DATASET / "test" / "sample.h5") as f:
    reference = f["data"][:].reshape(-1, 4)
generated = []
for filename in sorted(RESULTS.glob(f"generated_{TAG}_*.h5")):
    with h5py.File(filename) as f:
        generated.append(f["data"][:].reshape(-1, 4))
assert generated, "Run generation first"
generated = np.concatenate(generated)
assert generated.shape == reference.shape and np.isfinite(generated).all()
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    lo = min(reference[:, i].min(), generated[:, i].min(), MEAN[i] - 4 * STD[i])
    hi = max(reference[:, i].max(), generated[:, i].max(), MEAN[i] + 4 * STD[i])
    bins = np.linspace(lo, hi, 51)
    ax.hist(reference[:, i], bins=bins, density=True, histtype="step", label="Reference test")
    ax.hist(generated[:, i], bins=bins, density=True, histtype="step", label="Generated")
    grid = np.linspace(lo, hi, 300)
    density = np.exp(-0.5 * ((grid - MEAN[i]) / STD[i])**2) / (np.sqrt(2 * np.pi) * STD[i])
    ax.plot(grid, density, "k--", label="Target Gaussian")
    ax.set(xlabel=f"Feature {i}", ylabel="Density")
    ax.legend()
plt.show()
print("Target mean:", MEAN, "Generated:", generated.mean(0))
print("Target std: ", STD, "Generated:", generated.std(0))